In [ ]:
import pickle
import socket
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

np.random.seed(0)
torch.manual_seed(0)

sys.path.insert(0, str(Path.cwd().parent))

from fig_utils.basii import compute_orthogonal_subspace
from fig_utils.perturbation import (
    compute_perturbation_distance_stats,
    generate_w_perturb_x,
    perturbation_goal_bounds,
    position_latent_indices,
)
from fig_utils.plots import (
    plot_basis_3d_trajectories,
    plot_boxplot_by_group,
    plot_perturbation_latent_snapshots_attractor,
)
from fig_utils.transformed_rnn import transformed_rnn
from vi_rnn.data_utils import make_all_trials, stim_end_bins
from vi_rnn.generate import generate
from vi_rnn.saving import load_model, CPU_Unpickler
from vi_rnn.utils import get_orth_proj_latents

%matplotlib inline

In [ ]:
cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=6))

hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/synthetic")
    data_root = Path("/home/matthijs/swm_rnn/data")
else:
    out_dir = Path("/Users/matthijs/swm_rnn/final_models/synthetic")
    data_root = Path.cwd().parent / "data"


model_dirs = [
    "transient_low_rank_one_to_one_dim_z_3_date_2026_06_11_T_20_26_27",
    "transient_low_rank_one_to_one_dim_z_3_date_2026_06_11_T_20_26_22",
    "transient_low_rank_one_to_one_dim_z_3_date_2026_06_12_T_01_39_11",
    "transient_low_rank_one_to_one_dim_z_20_date_2026_06_08_T_22_51_44",
    "transient_low_rank_one_to_one_dim_z_20_date_2026_06_09_T_16_21_21",
    "transient_low_rank_one_to_one_dim_z_20_date_2026_06_08_T_16_46_44",
    "transient_low_rank_one_to_one_dim_z_40_date_2026_06_08_T_22_51_43",
    "transient_low_rank_one_to_one_dim_z_40_date_2026_06_08_T_16_46_47",
    "transient_low_rank_one_to_one_dim_z_40_date_2026_06_09_T_19_32_31",
]

In [ ]:
# --- controls ---
P = 6
n_duplicates = 10
n_pcs_time_basii = 1
n_repeats_pert = 100
noise_scale_pert = 1.0
bin_size = 0.05

BINS_AFTER_LAST_STIM = 9
N_BINS_FOR_R = 15
time_windows = [[20, 60], [20, 60]]  # nb10 BASII windows
# t_perturb / r window set from make_all_trials u in loop setup below
highlight_cond = 2

generate_plots = True
run = True

In [ ]:
def trial_indices_by_condition(labels, n_conds, n_repeats):
    inds = []
    for p in range(n_conds):
        pool = np.where(labels == p)[0]
        if len(pool) < n_repeats:
            raise ValueError(
                f"condition {p}: need {n_repeats} trials, have {len(pool)}"
            )
        inds.extend(np.random.choice(pool, size=n_repeats, replace=False))
    return np.array(inds, dtype=int)

In [ ]:
if run:
    rows = []

    task_params_file = Path(str(out_dir / model_dirs[0]) + "_task_params.pkl")
    with open(task_params_file, "rb") as f:
        task_params = CPU_Unpickler(f).load()

    u, _, labels_training, _ = make_all_trials(
        task_params,
        dur=8,
        n_stim=P,
        n_pos=1,
        cue_dur=-1,
        bin_size=bin_size,
        interval_dur="mean",
        delay_dur="mean",
    )
    u = torch.tensor(u, dtype=torch.float32)
    labels_training = np.int_(labels_training)

    last_stim_end = int(stim_end_bins(u.detach().cpu().numpy()).max())
    t_perturb = last_stim_end + BINS_AFTER_LAST_STIM
    t_move_start = t_perturb
    t_move_end = t_move_start + N_BINS_FOR_R
    plot_ts = [t_perturb - 2, t_perturb, t_perturb + 10, t_move_end]

    for model_name in model_dirs:
        model_path = out_dir / model_name
        print("Loading", model_path)

        vae, training_params, _ = load_model(
            str(model_path), load_encoder=False, backward_compat=False
        )
        print(
            "dim_z:",
            vae.dim_z,
            "| centroid_loss:",
            training_params.get("centroid_loss_weight"),
        )

        Zo, _, _, _ = generate(
            vae,
            u=u,
            x=None,
            k=n_duplicates,
            noise_scale=1.0,
            initial_state="prior_mean",
        )
        Zo_np = Zo.detach().cpu().numpy()
        projection_matrix = get_orth_proj_latents(vae).cpu().numpy()
        Z = np.einsum("ij,bjtk->kbit", projection_matrix, Zo_np)

        z_dpca = Z.mean(axis=0).transpose(1, 0, 2)
        n_steps = z_dpca.shape[-1]
        n_pcs_time = n_pcs_time_basii

        if vae.dim_z > 2:
            (
                transform,
                _,
                _,
                W_full,
                global_mean,
                scaling,
            ) = compute_orthogonal_subspace(
                z_dpca,
                marg_order=["t", "s1"],
                n_components=[n_pcs_time_basii, 2],
                time_windows=time_windows,
                center=True,
                center_marginals=True,
                soft_norm_constant=5.0,
                orthogonalize=True,
                standardize=False,
                basis="pca",
                lrr_reg=1e-5,
            )
            Z_basii = transform(z_dpca)
            n_pcs_time = 1
            scaling_safe = np.where(scaling == 0, 1.0, scaling)
            A_pert = W_full.T @ np.diag(1 / scaling_safe) @ projection_matrix
            b_pert = W_full.T @ np.diag(1 / scaling_safe) @ global_mean
        else:
            Z_basii = z_dpca
            n_pcs_time = 0
            A_pert = projection_matrix
            b_pert = np.zeros(vae.dim_z)

        Z_T = Z_basii.transpose(1, 0, 2)
        traj_labels = np.arange(P)[:, None]

        if generate_plots:
            plot_basis_3d_trajectories(
                Z_T,
                traj_labels,
                pos_ind=0,
                cmap=cmap,
                n_pcs_time=n_pcs_time,
                bin_size=bin_size,
                plt_start=0,
                plt_end=n_steps,
                jupyter_backend="static",
                window_size=(2000, 1000),
                show=True,
                xscale=1,
                yscale=1.5,  # time
                zscale=1,
                tscale=2,
                stim_mark_times=[0.3],
                tube_radius=0.05,
                azimuth=5,
                elevation=-30,
                zoom=1.75,
                x_axis_scale=0.7,
                y_axis_scale=1,
                z_axis_scale=1,
                axis_line_width=8,
                stim_line_width=4,
                shadow_opacity=0.2,
                shadow_line_width=10,
            )

        rnn_proj = transformed_rnn(vae, A_pert, b_pert)
        rnn_proj.z0 = np.zeros(rnn_proj.dim_z)

        if vae.dim_z > n_pcs_time + 1:
            z1, z2 = position_latent_indices(n_pcs_time, pos=0)
        else:
            z1, z2 = 0, min(1, vae.dim_z - 1)

        u_pert = np.repeat(u.detach().cpu().numpy(), n_repeats_pert, axis=0)
        Z_base = generate_w_perturb_x(rnn_proj, u=u_pert, noise_scale=noise_scale_pert)
        pert_z1 = perturbation_goal_bounds(Z_base[:, z1, t_perturb])
        pert_z2 = perturbation_goal_bounds(Z_base[:, z2, t_perturb])
        goals = np.random.rand(u_pert.shape[0], 2)
        goals[:, 0] = goals[:, 0] * (pert_z1[1] - pert_z1[0]) + pert_z1[0]
        goals[:, 1] = goals[:, 1] * (pert_z2[1] - pert_z2[0]) + pert_z2[0]

        Z_pert = generate_w_perturb_x(
            rnn_proj,
            u=u_pert,
            noise_scale=noise_scale_pert,
            perturb_at_t=t_perturb,
            perturb_weights=rnn_proj.W2,
            perturb_inds=[z1, z2],
            goal=goals,
            goal_amp=1.0,
        )
        labels_pos = np.repeat(np.arange(P), n_repeats_pert)

        if generate_plots:
            plot_perturbation_latent_snapshots_attractor(
                Z_base,
                Z_pert,
                labels_pos,
                z1=z1,
                z2=z2,
                cmap=cmap,
                plot_ts=plot_ts,
                highlight_cond=highlight_cond,
                bin_size=bin_size,
                show=True,
            )

        stats = compute_perturbation_distance_stats(
            Z_base,
            Z_pert,
            labels_pos,
            z1=z1,
            z2=z2,
            t_move_start=t_move_start,
            t_move_end=t_move_end,
        )
        print("perturbation r =", stats["pearson_r"])

        rows.append(
            {
                "name": model_name,
                "path": str(model_path),
                "dim_z": vae.dim_z,
                "centroid_loss": training_params.get("centroid_loss_weight"),
                "macaque": f"dz{vae.dim_z}",
                "pearson_r": stats["pearson_r"],
                "p_val": stats["p_val"],
                "slope": stats["slope"],
                "n_valid_trials": stats["n_valid_trials"],
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    pickle.dump(df, open("../data/processed/df_perturb_transient_student.pkl", "wb"))
else:
    df = pickle.load(open("../data/processed/df_perturb_transient_student.pkl", "rb"))

In [ ]:
# Summary boxplot across models (notebook 05 style)
order = [str(d) for d in sorted(df["dim_z"].unique())]
plot_df = df.copy()
plot_df["dim_z_str"] = plot_df["dim_z"].astype(str)

plot_boxplot_by_group(
    plot_df,
    group_col="dim_z_str",
    value_col="pearson_r",
    order=order,
    ylims=(-0.5, 1),
    box_w=1.0,
    box_h=1,
)

df